In [ ]:

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score

df = pd.read_csv(r"C:\Users\Tanuja\Downloads\california_housing.csv")

df.columns = df.columns.astype(str).str.strip()
df = df.loc[:, ~df.columns.str.lower().str.startswith("unnamed")]

target_col = None

for col in df.columns:
    if col.lower().replace("_", "").replace(" ", "") in [
        "medhouseval", "medianhousevalue", "target"
    ]:
        target_col = col
        break

if target_col is None:
    raise ValueError("Target column not found. Check dataset columns.")

X = df.drop(columns=[target_col]).select_dtypes(include="number")
y = pd.to_numeric(df[target_col], errors="coerce")

valid_rows = X.notna().all(axis=1) & y.notna()
X = X.loc[valid_rows]
y = y.loc[valid_rows]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

threshold = y_train.median()

y_train = (y_train > threshold).astype(int)
y_test = (y_test > threshold).astype(int)

linear_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="linear"))
])

rbf_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf"))
])

linear_model.fit(X_train, y_train)
rbf_model.fit(X_train, y_train)

linear_pred = linear_model.predict(X_test)
rbf_pred = rbf_model.predict(X_test)

print("Median House Value Threshold:", threshold)

print("\nLinear Kernel Confusion Matrix:")
print(confusion_matrix(y_test, linear_pred, labels=[0, 1]))

print("\nNonlinear RBF Kernel Confusion Matrix:")
print(confusion_matrix(y_test, rbf_pred, labels=[0, 1]))

results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Linear Kernel": [
        accuracy_score(y_test, linear_pred),
        precision_score(y_test, linear_pred, zero_division=0),
        recall_score(y_test, linear_pred, zero_division=0),
        f1_score(y_test, linear_pred, zero_division=0)
    ],
    "RBF Kernel": [
        accuracy_score(y_test, rbf_pred),
        precision_score(y_test, rbf_pred, zero_division=0),
        recall_score(y_test, rbf_pred, zero_division=0),
        f1_score(y_test, rbf_pred, zero_division=0)
    ]
})

print("\nKernel Performance Comparison:")
print(results.to_string(index=False))


In [ ]:
%pip install pandas scikit-learn
# 1. AIM

To implement a Support Vector Machine classifier using a Nonlinear Kernel on a suitable open-access dataset and compare its classification performance with the Linear Kernel.

# 2. DATASET USED

**Dataset Name:** California Housing Dataset

**Dataset Path:** `C:\Users\Tanuja\Downloads\california_housing.csv`

The California Housing dataset contains housing-related and demographic information about different districts in California.

The standard dataset contains 20,640 records, 8 numerical input features and one continuous target variable.

### Features of the Dataset

| Feature | Description |
|---|---|
| MedInc | Median income in the block group |
| HouseAge | Median house age |
| AveRooms | Average number of rooms per household |
| AveBedrms | Average number of bedrooms per household |
| Population | Population of the block group |
| AveOccup | Average number of household members |
| Latitude | Geographical latitude |
| Longitude | Geographical longitude |
| MedHouseVal | Median house value (target) |

**Target Variable Conversion:**

The original California Housing dataset is a regression dataset because its target variable contains continuous numerical values.

To apply SVM classification, the target variable is converted into two classes using the median calculated from the training target.

- 0 – Low house value (less than or equal to the training median).
- 1 – High house value (greater than the training median).

This makes the dataset suitable for binary classification without modifying the original CSV file.

**Why is this dataset used?**

1. It represents a real-world housing problem.
2. It contains multiple numerical input features.
3. It can be converted into a binary classification problem.
4. It is suitable for comparing Linear and Nonlinear SVM kernels.
5. It allows evaluation using accuracy, precision, recall and F1-score.

# 3. THEORY

## A. Support Vector Machine (SVM)

Support Vector Machine is a supervised machine learning algorithm used for classification and regression problems.

It is mainly used for classification by finding an optimal decision boundary that separates different classes.

The main objective of SVM is to find a hyperplane that maximizes the margin between different classes.

SVM is effective for datasets containing multiple numerical features and can handle both linear and nonlinear classification problems using different kernel functions.

## B. Nonlinear Support Vector Machine

A Nonlinear Support Vector Machine is used when the classes cannot be separated effectively using a straight-line decision boundary or linear hyperplane.

It uses nonlinear kernel functions to identify complex relationships between data points.

In this practical, the Radial Basis Function (RBF) Kernel is used to perform nonlinear classification.

## C. Kernel Function

A Kernel Function measures the similarity between two data points and helps SVM construct a suitable decision boundary.

Kernel functions allow SVM to learn nonlinear relationships without explicitly calculating the transformation into a higher-dimensional feature space.

Common types of kernel functions are:

1. Linear Kernel.
2. Polynomial Kernel.
3. RBF (Radial Basis Function) Kernel.
4. Sigmoid Kernel.

In this practical, Linear and RBF kernels are compared.

## D. Radial Basis Function (RBF) Kernel

RBF stands for Radial Basis Function.

It is a nonlinear kernel used in SVM to classify data points by considering their similarity or distance.

The RBF kernel can create curved and complex decision boundaries.

### Mathematical Formula

K(Xᵢ, Xⱼ) = exp(-γ ||Xᵢ - Xⱼ||²)

Where:

- Xᵢ = First data point.
- Xⱼ = Second data point.
- γ (Gamma) = Kernel parameter.
- ||Xᵢ - Xⱼ||² = Squared Euclidean distance.
- exp = Exponential function.

The RBF kernel measures similarity based on the distance between two data points.

## E. Difference Between Linear and Nonlinear Kernel

**Linear Kernel:**

The Linear Kernel creates a straight-line decision boundary in two dimensions or a hyperplane in higher dimensions.

Formula:

K(Xᵢ, Xⱼ) = XᵢᵀXⱼ

**RBF Kernel:**

The RBF Kernel creates nonlinear decision boundaries and can handle complex relationships between features.

| Linear Kernel | RBF Kernel |
|---|---|
| Creates a linear decision boundary | Creates a nonlinear decision boundary |
| Uses the dot product | Uses distance-based similarity |
| Suitable for approximately linearly separable data | Suitable for complex nonlinear data |
| Generally simpler | More flexible |
| Does not have a gamma parameter | Uses a gamma parameter |

## F. Working of SVM with RBF Kernel

SVM with RBF Kernel works through the following steps:

1. Load the dataset.
2. Select input features and the target variable.
3. Convert the continuous target into binary classes.
4. Divide the dataset into training and testing sets.
5. Standardize the input features.
6. Apply the RBF Kernel to the training data.
7. Find the decision boundary using the SVM algorithm.
8. Predict the classes for testing data.
9. Repeat the classification process using the Linear Kernel.
10. Compare both models using evaluation metrics.

## G. Hyperplane

A Hyperplane is a decision boundary that separates different classes in a feature space.

In two dimensions, it is a straight line.

In three dimensions, it is a plane.

In higher dimensions, it is called a hyperplane.

SVM identifies an optimal hyperplane to separate different classes.

## H. Support Vectors

Support Vectors are the data points closest to the decision boundary or located within the margin boundaries.

They are important because they determine the position and orientation of the separating hyperplane.

## I. Margin

Margin is the distance between the decision boundary and the nearest data points from different classes.

SVM tries to maximize the margin to improve its ability to classify unseen observations.

For a standard linear SVM, the margin width is:

Margin = 2 / ||w||

Where w is the weight vector.

## J. C Parameter in SVM

C is the regularization parameter that controls the balance between maximizing the margin and minimizing classification errors.

- Small C: Allows more classification errors and generally produces stronger regularization.
- Large C: Penalizes classification errors more strongly and generally produces a narrower margin.

In this practical, the default value C = 1.0 is used for both kernels.

## K. Gamma Parameter in RBF Kernel

Gamma is a parameter that controls the influence of individual training observations in the RBF Kernel.

- Low Gamma: Each training point has a wider influence, resulting in smoother decision boundaries.
- High Gamma: Each training point has a more localized influence, resulting in more complex decision boundaries.

In this practical, Scikit-learn's default gamma = "scale" is used for the RBF Kernel.

## L. Feature Scaling

Feature Scaling is the process of transforming numerical attributes into a comparable scale.

SVM is sensitive to feature magnitudes. Features with larger numerical ranges may influence the decision boundary more strongly.

In this practical, StandardScaler is used.

Formula:

Z = (X - μ) / σ

Where:

- X = Original value.
- μ = Mean of the feature.
- σ = Standard deviation.
- Z = Standardized value.

StandardScaler is fitted on training data and the same transformation is applied to testing data.

## M. Binary Classification using California Housing

The original California Housing dataset contains continuous house values.

SVM classification requires categorical class labels. Therefore, the continuous target is converted into two categories using the median house value calculated from training data.

The conversion is:

- If house value is greater than the training median, class 1 is assigned.
- If house value is less than or equal to the training median, class 0 is assigned.

The same threshold is used for both training and testing targets.

This creates a binary classification problem.

## N. Classification Evaluation Metrics

The following evaluation metrics are used to compare the Linear and RBF kernels.

### 1. Confusion Matrix

A Confusion Matrix compares actual class labels with predicted class labels.

| Actual / Predicted | Predicted 0 | Predicted 1 |
|---|---|---|
| Actual 0 | True Negative (TN) | False Positive (FP) |
| Actual 1 | False Negative (FN) | True Positive (TP) |

**True Positive (TP):** Correctly predicts high house value.

**True Negative (TN):** Correctly predicts low house value.

**False Positive (FP):** Predicts high house value, but actual value belongs to the low class.

**False Negative (FN):** Predicts low house value, but actual value belongs to the high class.

### 2. Accuracy

Accuracy measures the proportion of correctly classified observations.

Formula:

Accuracy = (TP + TN) / (TP + TN + FP + FN)

### 3. Precision

Precision measures how many observations predicted as positive are actually positive.

Formula:

Precision = TP / (TP + FP)

### 4. Recall

Recall measures how many actual positive observations are correctly identified by the model.

Formula:

Recall = TP / (TP + FN)

### 5. F1-Score

F1-score is the harmonic mean of precision and recall.

Formula:

F1 = 2 × (Precision × Recall) / (Precision + Recall)

## O. Advantages of Nonlinear SVM

1. Can identify complex nonlinear relationships.
2. Can create flexible decision boundaries.
3. Supports high-dimensional input data.
4. Can classify data that is not linearly separable in the original feature space.
5. The RBF kernel is useful for many nonlinear classification problems.

## P. Limitations of Nonlinear SVM

1. It can require more computational time than a Linear Kernel.
2. Performance depends on selecting suitable C and Gamma values.
3. It requires feature scaling.
4. It can overfit when the kernel parameters are not selected properly.
5. It can be expensive for large datasets.

# 4. ALGORITHM

1. Start.
2. Import Pandas and required Scikit-learn libraries.
3. Load the California Housing dataset using read_csv().
4. Identify the target variable.
5. Separate numerical input features X and target variable y.
6. Remove rows containing missing input or target values.
7. Divide the dataset into training and testing sets.
8. Calculate the median of the training target variable.
9. Convert the training and testing targets into binary classes using the training median.
10. Create an SVM model using the Linear Kernel.
11. Apply StandardScaler to the input features.
12. Train the Linear Kernel SVM model.
13. Create another SVM model using the RBF Kernel.
14. Apply StandardScaler and train the RBF Kernel SVM model.
15. Predict output values using both models.
16. Generate confusion matrices for both kernels.
17. Calculate accuracy, precision, recall and F1-score.
18. Compare the classification performance of both kernels.
19. Stop.

# 5. DETAILED EXPLANATION OF EACH LINE OF CODE

## A. Importing Libraries

**1. import pandas as pd**

Imports Pandas for loading and manipulating the dataset.

**2. from sklearn.model_selection import train_test_split**

Imports train_test_split() to divide the dataset into training and testing sets.

**3. from sklearn.preprocessing import StandardScaler**

Imports StandardScaler to standardize numerical features.

**4. from sklearn.pipeline import Pipeline**

Imports Pipeline to combine feature scaling and SVM classification into one workflow.

**5. from sklearn.svm import SVC**

Imports the Support Vector Classifier used to perform classification.

**6. from sklearn.metrics import confusion_matrix, accuracy_score**

Imports functions for generating the confusion matrix and calculating accuracy.

**7. from sklearn.metrics import precision_score, recall_score, f1_score**

Imports the functions used to calculate precision, recall and F1-score.

## B. Loading the Dataset

`df = pd.read_csv(r"C:\Users\Tanuja\Downloads\california_housing.csv")`

- read_csv() reads the CSV file.
- df stores the dataset in a Pandas DataFrame.
- The specified path identifies the location of the dataset.

## C. Selecting the Target Variable

`target_col = "MedHouseVal"`

Specifies the name of the target column in the California Housing dataset.

## D. Selecting Input and Output Variables

`X = df.drop(columns=[target_col])`

Removes the target column from the dataset and stores the remaining input features in X.

`y = df[target_col]`

Selects the MedHouseVal column and stores it in y.

`X = X.select_dtypes(include="number")`

Selects numerical input features for SVM classification.

## E. Splitting the Dataset

`train_test_split(X, y, test_size=0.2, random_state=42)`

Divides the dataset into:

- 80% training data.
- 20% testing data.

random_state=42 ensures reproducible splitting.

## F. Converting the Target into Binary Classes

`threshold = y_train.median()`

Calculates the median house value from the training target values.

`y_train = (y_train > threshold).astype(int)`

Converts training target values into binary classes.

- Values greater than the median become 1.
- Values less than or equal to the median become 0.

`y_test = (y_test > threshold).astype(int)`

Converts testing target values using the same training median.

This ensures that the testing data does not influence the threshold calculation.

## G. Creating Linear Kernel SVM

`linear_model = Pipeline([...])`

Creates a pipeline containing StandardScaler and SVC.

`StandardScaler()`

Standardizes the numerical input features.

`SVC(kernel="linear")`

Creates an SVM classifier using the Linear Kernel.

The model attempts to find a linear decision boundary that separates the two classes.

## H. Creating Nonlinear RBF Kernel SVM

`rbf_model = Pipeline([...])`

Creates another pipeline containing StandardScaler and SVC.

`SVC(kernel="rbf")`

Creates an SVM classifier using the Radial Basis Function Kernel.

The RBF kernel can create nonlinear decision boundaries to classify complex data.

## I. Training Both Models

`linear_model.fit(X_train, y_train)`

Trains the Linear Kernel SVM using the training dataset.

`rbf_model.fit(X_train, y_train)`

Trains the Nonlinear RBF Kernel SVM using the same training dataset.

The pipelines fit the scalers on training data before fitting their respective SVM classifiers.

## J. Making Predictions

`linear_pred = linear_model.predict(X_test)`

Predicts the binary class labels using the Linear Kernel.

`rbf_pred = rbf_model.predict(X_test)`

Predicts the binary class labels using the RBF Kernel.

## K. Generating Confusion Matrices

`confusion_matrix(y_test, linear_pred)`

Compares actual and predicted values for the Linear Kernel.

`confusion_matrix(y_test, rbf_pred)`

Compares actual and predicted values for the RBF Kernel.

The matrix shows True Positive, True Negative, False Positive and False Negative values.

## L. Comparing Classification Performance

`accuracy_score(y_test, linear_pred)`

Calculates the accuracy of the Linear Kernel model.

`accuracy_score(y_test, rbf_pred)`

Calculates the accuracy of the RBF Kernel model.

`precision_score()`

Calculates the precision of each model.

`recall_score()`

Calculates the recall of each model.

`f1_score()`

Calculates the F1-score of each model.

## M. Displaying Results

`pd.DataFrame({...})`

Creates a comparison table containing the four evaluation metrics for both kernels.

`results.to_string(index=False)`

Displays the comparison table without row indices.

# 6. EXPECTED OUTPUT

For the standard California Housing dataset:

- Total records: 20,640.
- Numerical input features: 8.
- Original target variable: MedHouseVal.
- Training records: 16,512.
- Testing records: 4,128.
- Number of classification classes: 2.
- Kernels compared: Linear and RBF.

The output displays:

1. Linear Kernel Confusion Matrix.
2. Nonlinear RBF Kernel Confusion Matrix.
3. Accuracy comparison.
4. Precision comparison.
5. Recall comparison.
6. F1-score comparison.

Actual metric values depend on the CSV file and the execution results.

# 7. ORAL EXAMINATION – VIVA QUESTIONS AND ANSWERS

## Basic Questions

**Q1. What is SVM?**

Support Vector Machine is a supervised machine learning algorithm mainly used for classification and also applicable to regression.

**Q2. What is a Nonlinear Kernel?**

A Nonlinear Kernel allows SVM to create nonlinear decision boundaries for separating complex data.

**Q3. Which kernel is used in this practical?**

RBF (Radial Basis Function) Kernel.

**Q4. What is RBF?**

RBF stands for Radial Basis Function.

**Q5. What is a Kernel Function?**

A kernel function measures similarity between data points and helps SVM construct a decision boundary.

**Q6. What is the main objective of SVM?**

To identify an optimal decision boundary that maximizes the margin between classes while controlling classification errors.

**Q7. What is a Hyperplane?**

A hyperplane is a decision boundary that separates different classes in a feature space.

**Q8. What are Support Vectors?**

Support Vectors are data points closest to the decision boundary or lying within the margin boundaries.

**Q9. What is Margin?**

Margin is the distance between the separating hyperplane and its nearest data points.

**Q10. What is the difference between Linear and RBF Kernel?**

Linear Kernel creates a linear decision boundary, whereas RBF Kernel creates a nonlinear decision boundary.

**Q11. What is Gamma?**

Gamma controls how far the influence of individual training samples extends in the RBF Kernel.

**Q12. What is C in SVM?**

C is the regularization parameter that controls the trade-off between margin size and classification errors.

# 8. DIFFICULT EXTERNAL EXAMINER QUESTIONS AND ANSWERS

**Q13. Why is RBF Kernel called a nonlinear kernel?**

Because it can produce curved and complex decision boundaries instead of restricting classification to a straight line or hyperplane in the original feature space.

**Q14. What is the mathematical formula of RBF Kernel?**

K(Xᵢ, Xⱼ) = exp(-γ ||Xᵢ - Xⱼ||²)

**Q15. What happens when Gamma increases?**

A higher gamma gives each training point a more localized influence and can create more complex decision boundaries. Very high gamma may cause overfitting.

**Q16. What happens when Gamma decreases?**

A lower gamma gives each training point a wider influence and generally creates smoother decision boundaries. Very low gamma may cause underfitting.

**Q17. What is the difference between C and Gamma?**

C controls the penalty for classification errors, whereas Gamma controls the influence of training points in the RBF Kernel.

**Q18. Why are both kernels trained on the same dataset?**

To provide a fair comparison of their classification performance using the same training and testing observations.

**Q19. Why is StandardScaler applied before SVM?**

SVM is sensitive to feature magnitudes. StandardScaler brings the numerical features onto a comparable scale.

**Q20. Can RBF Kernel handle nonlinear data?**

Yes. RBF Kernel can model complex nonlinear relationships between input features and target classes.

**Q21. Is RBF always better than Linear Kernel?**

No. The better kernel depends on the dataset and the decision boundary required. RBF can perform better on nonlinear data, while Linear Kernel may be more suitable for approximately linearly separable data.

**Q22. What happens if C is very high?**

The model penalizes training classification errors more strongly and may produce a narrower margin, increasing the possibility of overfitting.

**Q23. What happens if C is very low?**

The model allows more classification errors and applies stronger regularization, possibly causing underfitting.

**Q24. What is the Kernel Trick?**

The Kernel Trick calculates similarities between data points as if they had been transformed into a higher-dimensional feature space, without explicitly calculating the transformation.

**Q25. Why is California Housing converted into binary classes?**

Because the original dataset has a continuous target variable, while this practical requires classification. The median threshold converts house values into low and high categories.

**Q26. Why is the training median used as the threshold?**

To avoid using information from the testing target distribution while defining the classification classes.

**Q27. What is the difference between SVC and SVR?**

SVC performs classification, whereas SVR performs regression.

**Q28. What is the limitation of RBF Kernel on large datasets?**

RBF Kernel SVM can require substantial computational time and memory for large training datasets.

**Q29. What is Data Leakage?**

Data Leakage occurs when information from testing data is used during training or preprocessing, resulting in misleading evaluation.

**Q30. How can Data Leakage be prevented in this practical?**

By splitting the dataset first and fitting StandardScaler only on the training data through Pipeline.

# 9. DATASET-RELATED QUESTIONS

**Q1. What is the California Housing dataset?**

It is a real-world dataset containing housing-related and demographic information about California districts.

**Q2. How many records are present in the standard dataset?**

20,640 records.

**Q3. How many numerical input features are present?**

Eight numerical input features.

**Q4. What is the original target variable?**

MedHouseVal.

**Q5. Is the original California Housing dataset a classification dataset?**

No. It is originally a regression dataset because the house-value target contains continuous numerical values.

**Q6. Why is the target converted into two classes?**

To make the dataset suitable for SVM classification.

**Q7. What does class 0 represent?**

House values less than or equal to the training-set median.

**Q8. What does class 1 represent?**

House values greater than the training-set median.

**Q9. What is the role of MedInc?**

It represents median income in the block group.

**Q10. Why is this dataset suitable after target conversion?**

It contains multiple numerical features and can be converted into a binary classification dataset for comparing Linear and Nonlinear SVM.

# 10. PRACTICAL EXAMINATION – CODING QUESTIONS

**Q1. How do you load the dataset?**

`pd.read_csv("california_housing.csv")`

**Q2. How do you select input features?**

`X = df.drop(columns=["MedHouseVal"])`

**Q3. How do you calculate the median?**

`y_train.median()`

**Q4. How do you convert continuous values into binary classes?**

`(y_train > threshold).astype(int)`

**Q5. How do you create Linear Kernel SVM?**

`SVC(kernel="linear")`

**Q6. How do you create RBF Kernel SVM?**

`SVC(kernel="rbf")`

**Q7. How do you standardize features?**

`StandardScaler()`

**Q8. How do you train the model?**

`model.fit(X_train, y_train)`

**Q9. How do you predict output?**

`model.predict(X_test)`

**Q10. How do you generate a confusion matrix?**

`confusion_matrix(y_test, y_pred)`

**Q11. How do you calculate Accuracy?**

`accuracy_score(y_test, y_pred)`

**Q12. How do you calculate Precision?**

`precision_score(y_test, y_pred)`

**Q13. How do you calculate Recall?**

`recall_score(y_test, y_pred)`

**Q14. How do you calculate F1-score?**

`f1_score(y_test, y_pred)`

# 11. IMPORTANT FORMULAS

**1. Linear Kernel**

K(Xᵢ, Xⱼ) = XᵢᵀXⱼ

**2. RBF Kernel**

K(Xᵢ, Xⱼ) = exp(-γ ||Xᵢ - Xⱼ||²)

**3. SVM Decision Boundary**

wᵀx + b = 0

**4. Margin Width**

Margin = 2 / ||w||

**5. Accuracy**

Accuracy = (TP + TN) / (TP + TN + FP + FN)

**6. Precision**

Precision = TP / (TP + FP)

**7. Recall**

Recall = TP / (TP + FN)

**8. F1-Score**

F1 = 2 × (Precision × Recall) / (Precision + Recall)

**9. Standardization**

Z = (X - μ) / σ

# 12. IMPORTANT POINTS FOR VIVA

1. SVM stands for Support Vector Machine.
2. RBF stands for Radial Basis Function.
3. RBF is a nonlinear kernel.
4. Linear Kernel creates a linear decision boundary.
5. RBF Kernel creates nonlinear decision boundaries.
6. Support Vectors determine the separating boundary.
7. C controls the regularization strength.
8. Gamma controls the influence of training points in the RBF Kernel.
9. StandardScaler is applied before SVM.
10. Both kernels use the same training and testing data.
11. California Housing is originally a regression dataset.
12. Its target is converted into binary classes using the training median.
13. Confusion Matrix, Accuracy, Precision, Recall and F1-score are used for comparison.